In [1]:
import os
import json
import string
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị sử dụng:", device)


Thiết bị sử dụng: cuda


In [ ]:
class PhoNERVocab:
    def __init__(self, train_path: str):
        self.pad = "<pad>"
        self.unk = "<unk>"

        train_data = [json.loads(line) for line in open(train_path, encoding='utf-8')]

        all_words = set()
        all_tags = set()
        for item in train_data:
            all_words.update(w.lower() for w in item["words"])
            all_tags.update(item["tags"])

        self.w2i = {self.pad: 0, self.unk: 1}
        for idx, word in enumerate(sorted(all_words), start=2):
            self.w2i[word] = idx
        self.i2w = {idx: word for word, idx in self.w2i.items()}

        sorted_tags = sorted(list(all_tags))
        self.tag2i = {tag: idx for idx, tag in enumerate(sorted_tags)}
        self.i2tag = {idx: tag for tag, idx in self.tag2i.items()}

    @property
    def n_tags(self) -> int:
        return len(self.tag2i)

    @property
    def len(self) -> int:
        return len(self.w2i)


In [6]:
class PhoNERDataset(Dataset):
    def __init__(self, path: str, vocab: PhoNERVocab):
        super().__init__()
        self.samples = [json.loads(line) for line in open(path, encoding='utf-8')]
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        item = self.samples[idx]
        unk_id = self.vocab.w2i["<unk>"]
        input_ids = torch.tensor([self.vocab.w2i.get(w.lower(), unk_id) for w in item["words"]], dtype=torch.long)
        tag_ids = torch.tensor([self.vocab.tag2i[t] for t in item["tags"]], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "tags": tag_ids
        }


def collate_fn_ner(items: list[dict]) -> dict:
    input_ids = [item["input_ids"] for item in items]
    tags = [item["tags"] for item in items]

    lengths = torch.tensor([len(ids) for ids in input_ids], dtype=torch.long)
    max_len = int(lengths.max())

    padded_inputs = torch.stack([
        F.pad(ids, pad=(0, max_len - len(ids)), mode="constant", value=0)
        for ids in input_ids
    ])
    padded_tags = torch.stack([
        F.pad(t, pad=(0, max_len - len(t)), mode="constant", value=-100)
        for t in tags
    ])

    return {
        "input_ids": padded_inputs,
        "lengths": lengths,
        "tags": padded_tags
    }


In [ ]:
class BiLstmNerModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        hidden_size: int = 256,
        n_layer: int = 5,
        n_tags: int = 20,
        padding_idx: int = 0,
        dropout: float = 0.2
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=hidden_size,
            padding_idx=padding_idx
        )

        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=n_layer,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layer > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            in_features=hidden_size * 2,
            out_features=n_tags
        )

    def forward(self, inputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded_feats = self.embedding(inputs)
        packed = pack_padded_sequence(
            embedded_feats,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_outputs, _ = self.bilstm(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        outputs = self.dropout(outputs)
        logits = self.classifier(outputs)

        return logits


In [10]:
def train_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device
) -> float:
    model.train()
    running_loss = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"]
        tags = batch["tags"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, lengths)
        loss = loss_fn(logits.view(-1, logits.shape[-1]), tags.view(-1))
        loss.backward()
        optimizer.step()

        running_loss.append(loss.item())

    return float(np.mean(running_loss))


@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device
) -> float:
    model.eval()
    gts = []
    preds = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"]
        tags = batch["tags"].to(device)

        logits = model(input_ids, lengths)
        pred_tags = logits.argmax(dim=-1)

        mask = (tags != -100)
        gts.extend(tags[mask].cpu().tolist())
        preds.extend(pred_tags[mask].cpu().tolist())

    f1 = f1_score(gts, preds, average="macro")
    return f1


In [11]:
import os

TRAIN_PATH = "datasets/PhoNER_COVID19/data/word/train_word.json"
DEV_PATH   = "datasets/PhoNER_COVID19/data/word/dev_word.json"
TEST_PATH  = "datasets/PhoNER_COVID19/data/word/test_word.json"

vocab = PhoNERVocab(train_path=TRAIN_PATH)

train_loader = DataLoader(PhoNERDataset(TRAIN_PATH, vocab), batch_size=32, shuffle=True, collate_fn=collate_fn_ner)
dev_loader   = DataLoader(PhoNERDataset(DEV_PATH, vocab),   batch_size=32, shuffle=False, collate_fn=collate_fn_ner)
test_loader  = DataLoader(PhoNERDataset(TEST_PATH, vocab),  batch_size=32, shuffle=False, collate_fn=collate_fn_ner)

model = BiLstmNerModel(
    vocab_size=vocab.len,
    hidden_size=256,
    n_layer=5,
    n_tags=vocab.n_tags,
    padding_idx=0,
    dropout=0.2
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100).to(device)

os.makedirs("checkpoints", exist_ok=True)
CHECKPOINT_PATH = "checkpoints/best_bilstm_ner.pth"

epoch = 0
best_f1 = 0.0
patience = 0
MAX_PATIENCE = 5

while True:
    epoch += 1
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
    dev_f1 = evaluate(model, dev_loader, device)

    print(f"Epoch [{epoch:02d}] | Train Loss: {train_loss:.4f} | Dev F1-macro: {dev_f1:.4f}")

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        patience = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"Best F1: {best_f1:.4f}")
    else:
        patience += 1
        print(f"Patience: {patience}/{MAX_PATIENCE}")

    if patience >= MAX_PATIENCE:
        break

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
test_f1 = evaluate(model, test_loader, device)

print(f"\nKẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG (BÀI 3 - BILSTM NER):")
print(f"Test Set Macro F1-score: {test_f1:.4f}")


Epoch [01] | Train Loss: 0.8969 | Dev F1-macro: 0.2258
Best F1: 0.2258
Epoch [02] | Train Loss: 0.2410 | Dev F1-macro: 0.4525
Best F1: 0.4525
Epoch [03] | Train Loss: 0.1109 | Dev F1-macro: 0.5813
Best F1: 0.5813
Epoch [04] | Train Loss: 0.0730 | Dev F1-macro: 0.6174
Best F1: 0.6174
Epoch [05] | Train Loss: 0.0495 | Dev F1-macro: 0.6706
Best F1: 0.6706
Epoch [06] | Train Loss: 0.0357 | Dev F1-macro: 0.7280
Best F1: 0.7280
Epoch [07] | Train Loss: 0.0326 | Dev F1-macro: 0.7375
Best F1: 0.7375
Epoch [08] | Train Loss: 0.0277 | Dev F1-macro: 0.7419
Best F1: 0.7419
Epoch [09] | Train Loss: 0.0204 | Dev F1-macro: 0.7742
Best F1: 0.7742
Epoch [10] | Train Loss: 0.0147 | Dev F1-macro: 0.7913
Best F1: 0.7913
Epoch [11] | Train Loss: 0.0107 | Dev F1-macro: 0.7859
Patience: 1/5
Epoch [12] | Train Loss: 0.0097 | Dev F1-macro: 0.7993
Best F1: 0.7993
Epoch [13] | Train Loss: 0.0098 | Dev F1-macro: 0.7897
Patience: 1/5
Epoch [14] | Train Loss: 0.0116 | Dev F1-macro: 0.7962
Patience: 2/5
Epoch [15] |